<a href="https://colab.research.google.com/github/crystalloide/Kafka/blob/main/kafka_4_1_installation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installation d'Apache Kafka 4.1.1 en Mode KRaft Standalone

Ce notebook installe et configure Apache Kafka 4.1 sur Linux (exempe : Ubuntu 24.04 en utilisant le mode KRaft (Kafka Raft Metadata), qui remplace désormais Zookeeper.

**Prérequis:**
- Ubuntu 24.04 avec accès sudo
- Minimum 2 GB de RAM
- Java 21 (sera installé)

## 1. Mise à jour du système et installation de Java

In [1]:
# Mettre à jour la liste des paquets
!sudo apt update -qq  2>/dev/null

# Installer OpenJDK 21
!sudo apt install -y openjdk-21-jdk -qq  2>/dev/null

# Vérifier l'installation de Java
!java -version

57 packages can be upgraded. Run 'apt list --upgradable' to see them.
openjdk-21-jdk is already the newest version (21.0.9+10-1~22.04).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Ubuntu-122.04, mixed mode, sharing)


## 2. Créer un utilisateur dédié et les répertoires Kafka

In [2]:
!sudo rm -rf /opt/kafka
!ls /opt/

google	nvidia


In [3]:
# Créer l'utilisateur système 'kafka' sans accès de connexion
!sudo useradd -r -m -U -d /opt/kafka -s /bin/false kafka 2>/dev/null || echo "L'utilisateur 'kafka' existe déjà"

# Créer le répertoire principal de Kafka
!sudo mkdir -p /opt/kafka

# Attribuer la propriété du répertoire à l'utilisateur kafka
!sudo chown -R kafka:kafka /opt/kafka

# Vérifier les permissions
!ls -ld /opt/kafka

L'utilisateur 'kafka' existe déjà
drwxr-xr-x 2 kafka kafka 4096 Nov 30 16:40 /opt/kafka


## 3. Télécharger et extraire Kafka 4.1.1

In [4]:
import os
import subprocess

# Se placer dans le répertoire temporaire
os.chdir('/tmp')

# Télécharger Kafka 4.1.1
print("Téléchargement de Kafka 4.1.1...")
!rm kafka_2.13-4.1.1.tgz
!wget -q https://downloads.apache.org/kafka/4.1.1/kafka_2.13-4.1.1.tgz
print("✓ Téléchargement terminé")

# Extraire l'archive
print("\nExtraction de l'archive...")
!sudo tar -xzf kafka_2.13-4.1.1.tgz -C /opt/kafka --strip-components=1
print("✓ Extraction terminée")

# Mettre à jour les permissions
!sudo chown -R kafka:kafka /opt/kafka

# Vérifier le contenu
print("\nContenu de /opt/kafka:")
!sudo ls -l /opt/kafka

Téléchargement de Kafka 4.1.1...
✓ Téléchargement terminé

Extraction de l'archive...
✓ Extraction terminée

Contenu de /opt/kafka:
total 64
drwxr-xr-x 3 kafka kafka  4096 Oct 26 18:50 bin
drwxr-xr-x 2 kafka kafka  4096 Oct 26 18:50 config
drwxr-xr-x 2 kafka kafka  4096 Nov 30 16:40 libs
-rw-r--r-- 1 kafka kafka 14707 Oct 26 18:49 LICENSE
drwxr-xr-x 2 kafka kafka  4096 Oct 26 18:50 licenses
-rw-r--r-- 1 kafka kafka 26819 Oct 26 18:49 NOTICE
drwxr-xr-x 2 kafka kafka  4096 Oct 26 18:50 site-docs


## 4. Configuration de Kafka en mode KRaft

In [5]:
# Lire le fichier de configuration actuel
print("Configuration actuelle de Kafka...\n")

# Vérifier les paramètres KRaft clés
!sudo grep -E "process.roles|node.id|controller.quorum" /opt/kafka/config/server.properties | head -5

Configuration actuelle de Kafka...

process.roles=broker,controller
# The node id associated with this instance's roles
node.id=1
controller.quorum.bootstrap.servers=localhost:9093
# Combined nodes (i.e. those with `process.roles=broker,controller`) must list the controller listener here at a minimum.


In [6]:
# Créer une sauvegarde du fichier de configuration original
!sudo cp /opt/kafka/config/server.properties /opt/kafka/config/server.properties.bak

# Configuration KRaft pour mode standalone
kraft_config = """# Mode KRaft - Broker et Controller sur le même nœud
process.roles=broker,controller
node.id=1

# Quorum Controller
controller.quorum.bootstrap.servers=localhost:9093
controller.quorum.voters=1@localhost:9093

# Listeners
listeners=PLAINTEXT://:9092,CONTROLLER://:9093
advertised.listeners=PLAINTEXT://localhost:9092,CONTROLLER://localhost:9093
controller.listener.names=CONTROLLER

# Protocoles de sécurité
listener.security.protocol.map=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT,SSL:SSL,SASL_PLAINTEXT:SASL_PLAINTEXT,SASL_SSL:SASL_SSL

# Répertoire de données
log.dirs=/opt/kafka/data

# Autres paramètres essentiels
offsets.topic.replication.factor=1
transaction.state.log.replication.factor=1
transaction.state.log.min.isr=1
"""

# Écrire la configuration dans un fichier temporaire
with open('/tmp/kraft_config.txt', 'w') as f:
    f.write(kraft_config)

# Ajouter la configuration au fichier server.properties
!sudo bash -c 'cat /tmp/kraft_config.txt >> /opt/kafka/config/server.properties'

print("✓ Configuration KRaft ajoutée")
print("\nVérification de la configuration:")
!sudo grep -E "process.roles|node.id|controller.quorum.voters|log.dirs" /opt/kafka/config/server.properties | tail -4

✓ Configuration KRaft ajoutée

Vérification de la configuration:
process.roles=broker,controller
node.id=1
controller.quorum.voters=1@localhost:9093
log.dirs=/opt/kafka/data


## 5. Générer un Cluster ID et formater le stockage

In [7]:
# Générer un Cluster ID unique
print("Génération du Cluster ID...\n")
result = !sudo /opt/kafka/bin/kafka-storage.sh random-uuid
cluster_id = str(result[0]).strip()
print(f"Cluster ID généré: {cluster_id}\n")

# Stocker le Cluster ID pour référence
with open('/tmp/cluster_id.txt', 'w') as f:
    f.write(cluster_id)

print("Cluster ID sauvegardé dans /tmp/cluster_id.txt")

Génération du Cluster ID...

Cluster ID généré: PPNOvqyZRjSm2NMXRjhv4g

Cluster ID sauvegardé dans /tmp/cluster_id.txt


In [8]:
# Lire le Cluster ID sauvegardé
with open('/tmp/cluster_id.txt', 'r') as f:
    cluster_id = f.read().strip()

print(f"Utilisation du Cluster ID: {cluster_id}\n")

# Formater le répertoire de stockage Kafka
print("Formatage du répertoire de stockage...\n")
!sudo /opt/kafka/bin/kafka-storage.sh format -t {cluster_id} -c /opt/kafka/config/server.properties

print("\n✓ Stockage formaté avec succès")

Utilisation du Cluster ID: PPNOvqyZRjSm2NMXRjhv4g

Formatage du répertoire de stockage...

Formatting metadata directory /opt/kafka/data with metadata.version 4.1-IV1.

✓ Stockage formaté avec succès


In [10]:
# Mettre à jour les permissions
!sudo chown -Rf kafka:kafka /opt/kafka

## 6. Créer un service systemd pour Kafka

In [12]:
# Contenu du fichier de service systemd
service_content = """[Unit]
Description=Apache Kafka Server
After=network.target

[Service]
Type=simple
User=kafka
ExecStart=/opt/kafka/bin/kafka-server-start.sh /opt/kafka/config/server.properties
ExecStop=/opt/kafka/bin/kafka-server-stop.sh
Restart=on-abnormal

[Install]
WantedBy=multi-user.target
"""

# Écrire le fichier de service
with open('/tmp/kafka.service', 'w') as f:
    f.write(service_content)

# Copier le fichier de service vers le répertoire systemd
!sudo cp /tmp/kafka.service /etc/systemd/system/kafka.service

# Vérifier le contenu du service
print("Contenu du service Kafka:")
!sudo cat /etc/systemd/system/kafka.service

Contenu du service Kafka:
[Unit]
Description=Apache Kafka Server
After=network.target

[Service]
Type=simple
User=kafka
ExecStart=/opt/kafka/bin/kafka-server-start.sh /opt/kafka/config/server.properties
ExecStop=/opt/kafka/bin/kafka-server-stop.sh
Restart=on-abnormal

[Install]
WantedBy=multi-user.target


## 7. Créer le répertoire de données et démarrer Kafka

In [13]:
# Créer le répertoire de données Kafka
print("Création du répertoire de données...")
!sudo mkdir -p /opt/kafka/data
!sudo chown -R kafka:kafka /opt/kafka/data

# Vérifier les permissions
!ls -ld /opt/kafka/data

print("\n✓ Répertoire de données créé")

Création du répertoire de données...
drwxr-xr-x 2 kafka kafka 4096 Nov 30 16:41 /opt/kafka/data

✓ Répertoire de données créé


In [31]:
# Recharger systemd pour reconnaître le nouveau service
#print("Rechargement de systemd...")
#!sudo systemctl daemon-reload

# Activer Kafka au démarrage du système
#print("Activation de Kafka au démarrage...")
#!sudo systemctl enable kafka

# Démarrer le service Kafka
#print("Démarrage du service Kafka...")
#!sudo systemctl restart kafka
!su kafka
!/opt/kafka/bin/kafka-server-start.sh /opt/kafka/config/server.properties > /tmp/kafka.log 2>&1 &

print("\n✓ Kafka démarré")


✓ Kafka démarré


In [29]:
# Pour arrêter :
!su kafka
!/opt/kafka/bin/kafka-server-stop.sh /opt/kafka/config/server.properties &

In [41]:
# Vérifier le statut de Kafka
print("Statut du service Kafka:\n")
!ps -aux | grep kafka
# !sudo systemctl status kafka

# Vérifier les ports de Kafka à l'écoute  :
!netstat -l | grep 909


Statut du service Kafka:

root       24569  9.6  2.9 3785148 393504 ?      Sl   16:50   0:15 java -Xmx1G -Xms1G -server -XX:+UseG1GC -XX:MaxGCPauseMillis=20 -XX:InitiatingHeapOccupancyPercent=35 -XX:+ExplicitGCInvokesConcurrent -XX:MaxInlineLevel=15 -Djava.awt.headless=true -Xlog:gc*:file=/opt/kafka/bin/../logs/kafkaServer-gc.log:time,tags:filecount=10,filesize=100M -Dcom.sun.management.jmxremote=true -Dcom.sun.management.jmxremote.authenticate=false -Dcom.sun.management.jmxremote.ssl=false -Dkafka.logs.dir=/opt/kafka/bin/../logs -Dlog4j2.configurationFile=/opt/kafka/bin/../config/log4j2.yaml -cp /opt/kafka/bin/../libs/activation-1.1.1.jar:/opt/kafka/bin/../libs/aopalliance-repackaged-3.0.6.jar:/opt/kafka/bin/../libs/argparse4j-0.7.0.jar:/opt/kafka/bin/../libs/caffeine-3.2.0.jar:/opt/kafka/bin/../libs/classgraph-4.8.179.jar:/opt/kafka/bin/../libs/commons-beanutils-1.11.0.jar:/opt/kafka/bin/../libs/commons-collections-3.2.2.jar:/opt/kafka/bin/../libs/commons-digester-2.1.jar:/opt/kafka/

## 8. Tests de Kafka

In [28]:
import time

# Attendre que Kafka soit complètement prêt
print("Attente du démarrage complet de Kafka (10 secondes)...")
time.sleep(10)

# Créer un topic de test
print("\nCréation d'un topic de test 'test-topic'...")
!su kafka
!/opt/kafka/bin/kafka-topics.sh --create --topic test-topic --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1

print("\n✓ Topic créé")

Attente du démarrage complet de Kafka (10 secondes)...

Création d'un topic de test 'test-topic'...
Created topic test-topic.

✓ Topic créé


In [33]:
# Lister les topics
print("Topics disponibles:\n")
!su kafka
!/opt/kafka/bin/kafka-topics.sh --list --bootstrap-server localhost:9092
# Autre façon :
#!sudo /opt/kafka/bin/kafka-topics.sh --list --bootstrap-server localhost:9092

Topics disponibles:

test-topic


In [34]:
# Afficher les détails du topic
print("Détails du topic 'test-topic':\n")
!sudo /opt/kafka/bin/kafka-topics.sh --describe --topic test-topic --bootstrap-server localhost:9092

Détails du topic 'test-topic':

Topic: test-topic	TopicId: p3SVPAg4QO2Kxo3jhCsUcQ	PartitionCount: 1	ReplicationFactor: 1	Configs: min.insync.replicas=1,segment.bytes=1073741824
	Topic: test-topic	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	LastKnownElr: 


## 9. Résumé et Informations Utiles

In [43]:
print("="*70)
print("INSTALLATION D'APACHE KAFKA 4.1.1 TERMINÉE")
print("="*70)

print("\n📍 CHEMINS IMPORTANTS:")
print("   - Installation: /opt/kafka")
print("   - Configuration: /opt/kafka/config/server.properties")
print("   - Données: /opt/kafka/data")
print("   - Service systemd: /etc/systemd/system/kafka.service")

print("\n⚙️  MODE DE FONCTIONNEMENT:")
print("   - Mode KRaft (Kafka Raft Metadata)")
print("   - Broker & Controller sur le même nœud")
print("   - Node ID: 1")

print("\n🔌 PORTS:")
print("   - Clients (PLAINTEXT): 9092")
print("   - Controller (interne): 9093")

print("\n📋 COMMANDES UTILES:")
print("   - Démarrer: sudo systemctl start kafka")
print("   - Arrêter: sudo systemctl stop kafka")
print("   - Statut: sudo systemctl status kafka")
print("   - Logs: sudo journalctl -u kafka -f")

print("\n📝 CRÉER UN TOPIC:")
print("   sudo /opt/kafka/bin/kafka-topics.sh --create \\")
print("     --topic mon-topic \\")
print("     --bootstrap-server localhost:9092 \\")
print("     --partitions 3 --replication-factor 1")

print("\n📤 PRODUIRE DES MESSAGES:")
print("   sudo /opt/kafka/bin/kafka-console-producer.sh \\")
print("     --topic mon-topic \\")
print("     --bootstrap-server localhost:9092")

print("\n📥 CONSOMMER DES MESSAGES:")
print("   sudo /opt/kafka/bin/kafka-console-consumer.sh \\")
print("     --topic mon-topic \\")
print("     --from-beginning \\")
print("     --bootstrap-server localhost:9092")

print("\n✅ Kafka 4.1.1 est prêt pour développement et tests!")
print("="*70)

INSTALLATION D'APACHE KAFKA 4.1.1 TERMINÉE

📍 CHEMINS IMPORTANTS:
   - Installation: /opt/kafka
   - Configuration: /opt/kafka/config/server.properties
   - Données: /opt/kafka/data
   - Service systemd: /etc/systemd/system/kafka.service

⚙️  MODE DE FONCTIONNEMENT:
   - Mode KRaft (Kafka Raft Metadata)
   - Broker & Controller sur le même nœud
   - Node ID: 1

🔌 PORTS:
   - Clients (PLAINTEXT): 9092
   - Controller (interne): 9093

📋 COMMANDES UTILES:
   - Démarrer: sudo systemctl start kafka
   - Arrêter: sudo systemctl stop kafka
   - Statut: sudo systemctl status kafka
   - Logs: sudo journalctl -u kafka -f

📝 CRÉER UN TOPIC:
   sudo /opt/kafka/bin/kafka-topics.sh --create \
     --topic mon-topic \
     --bootstrap-server localhost:9092 \
     --partitions 3 --replication-factor 1

📤 PRODUIRE DES MESSAGES:
   sudo /opt/kafka/bin/kafka-console-producer.sh \
     --topic mon-topic \
     --bootstrap-server localhost:9092

📥 CONSOMMER DES MESSAGES:
   sudo /opt/kafka/bin/kafka-consol